# Library

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime
from datetime import date
import os

# Question 1

In [2]:
df_orders = pd.read_csv("/Users/doanhaidang1703/Documents/AIO/datathon 2026/data/orders.csv")
df_orders["order_date"] = pd.to_datetime(df_orders["order_date"])
df_success_orders = df_orders[df_orders["order_status"] == "delivered"].copy()
#drop orders that are in the same day which affects the inter-order gap
df_success_orders = df_success_orders.drop_duplicates(subset = ["customer_id", "order_date"])
order_counts = df_success_orders.groupby("customer_id").size()
print(f" Number of customers: {len(order_counts)}")
returning_customers_id = order_counts[order_counts > 1].index
print(f" Number of returning customers: {len(returning_customers_id)}")

returning_customers_df = df_success_orders[df_success_orders["customer_id"].isin(returning_customers_id)].copy()
returning_customers_df



 Number of customers: 85115
 Number of returning customers: 62172


,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source
0,1,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search
2,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct
3,4,2012-07-04,59453,2360,delivered,credit_card,desktop,referral
4,6,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign
5,7,2012-07-06,57820,2886,delivered,credit_card,tablet,organic_search
...,...,...,...,...,...,...,...,...
646940,834372,2022-12-31,19490,33907,delivered,credit_card,mobile,email_campaign
646941,834377,2022-12-31,73046,37091,delivered,credit_card,mobile,referral
646942,834387,2022-12-31,107723,80516,delivered,credit_card,mobile,email_campaign
646943,834392,2022-12-31,139431,93510,delivered,paypal,desktop,direct


In [3]:
returning_customers_df = returning_customers_df.sort_values(by = ["customer_id", "order_date"])
returning_customers_df["prev_order_dates"] = returning_customers_df.groupby("customer_id")["order_date"].shift(1)
returning_customers_df

returning_customers_df["inter_order gap"] = (returning_customers_df["order_date"] - returning_customers_df["prev_order_dates"]).dt.days

median_gap = returning_customers_df["inter_order gap"].median()


In [4]:
print(f"The median of inter-order gap is {median_gap} days")

The median of inter-order gap is 178.0 days


# Question 2

In [5]:
df_products = pd.read_csv("/Users/doanhaidang1703/Documents/AIO/datathon 2026/data/products.csv")
df_products["gross_margin"] = (df_products["price"] - df_products["cogs"]) / df_products["price"]
avg_gross_margin = df_products.groupby("segment")["gross_margin"].mean()
print(avg_gross_margin)
print(f"Tỉ suất lợi nhuận gộp trung bình cao nhất là trong segment: {avg_gross_margin.idxmax()}")

segment
Activewear     0.265600
All-weather    0.284176
Balanced       0.258038
Everyday       0.236343
Performance    0.263650
Premium        0.285377
Standard       0.313442
Trendy         0.240758
Name: gross_margin, dtype: float64
Tỉ suất lợi nhuận gộp trung bình cao nhất là trong segment: Standard


# Question 3

In [6]:
df_returns = pd.read_csv("/Users/doanhaidang1703/Documents/AIO/datathon 2026/data/returns.csv")
df_returns
df_products_returns = pd.merge(df_returns, df_products, on = ["product_id"])

streetwear_returns = df_products_returns[df_products_returns["category"] == "Streetwear"].copy()
most_return_reason = streetwear_returns.groupby("return_reason").size().idxmax()
#df_products_returns
print(f"Return reason that appears the most is: {most_return_reason}")


Return reason that appears the most is: wrong_size


# Question 4

In [7]:
traffic_df = pd.read_csv("/Users/doanhaidang1703/Documents/AIO/datathon 2026/data/web_traffic.csv")
traffic_df
average_bounce_rate = traffic_df.groupby("traffic_source")["bounce_rate"].mean()
print(f"The traffic source with the lowest bounce rate is: {average_bounce_rate.idxmin()}")

The traffic source with the lowest bounce rate is: email_campaign


# Question 5

In [8]:
df_order_items = pd.read_csv("/Users/doanhaidang1703/Documents/AIO/datathon 2026/data/order_items.csv", 
    dtype={'promo_id': str, 'promo_id_2': str})

promo_order_df = df_order_items[df_order_items["promo_id"].notna()]
print(promo_order_df.head())
total_items = len(df_order_items)
promo_items = len(promo_order_df)
percentage_promo = (promo_items/ total_items) * 100
print(f"Percentage of rows with promotion {percentage_promo:.0f}%")


       order_id  product_id  quantity  unit_price  discount_amount  \
41316     46253        2250         6     1253.04          1127.74   
41317     46254        2251         7     1249.71          1312.20   
41319     46257         785         3      598.98           269.54   
41320     46257         786         2      611.66           183.50   
41321     46258        1093         6      604.71           544.24   

         promo_id promo_id_2  
41316  PROMO-0006        NaN  
41317  PROMO-0006        NaN  
41319  PROMO-0006        NaN  
41320  PROMO-0006        NaN  
41321  PROMO-0006        NaN  
Percentage of rows with promotion 39%


# Question 6

In [9]:
df_customers = pd.read_csv("/Users/doanhaidang1703/Documents/AIO/datathon 2026/data/customers.csv")
filtered_df_customers = df_customers[df_customers["age_group"].notna()].copy()
df_orders_success = df_orders[df_orders["order_status"] == "delivered"].copy()
#print(filtered_df_customers.head())
#have to reset_index as it returns a series with index customer_id so we have to make it a column to be able to merge
success_order_counts = df_orders_success.groupby("customer_id").size().reset_index(name = "order_count")
success_order_counts
merged_success = pd.merge(filtered_df_customers, success_order_counts, on = "customer_id", how = "left")
merged_success['order_count'] = merged_success['order_count'].fillna(0)
avg_orders_success = merged_success.groupby('age_group')['order_count'].mean().sort_values(ascending=False)
print(f"Age group with the most average orders: {avg_orders_success.idxmax()}")

Age group with the most average orders: 55+


# Question 7

In [16]:
df_region = pd.read_csv("/Users/doanhaidang1703/Documents/AIO/datathon 2026/data/geography.csv")
df_region_order = pd.merge(df_region, df_orders_success, on = "zip")
df_final = pd.merge(df_region_order, df_order_items, on = "order_id")
df_final["revenue"] = df_final["quantity"] * df_final["unit_price"]
total_revenue_region = df_final.groupby("region")["revenue"].sum()
answer = total_revenue_region.idxmax()
print(f"The region with the highest total revenue is: {answer}")
print(total_revenue_region)


The region with the highest total revenue is: East
region
Central    3.940034e+09
East       6.103315e+09
West       3.073743e+09
Name: revenue, dtype: float64


# Question 8

In [21]:
cancelled_orders = df_orders[df_orders["order_status"] == "cancelled"]
cancelled_orders
payment_ranking = cancelled_orders.groupby("payment_method").size()
highest_cancelled_payment = payment_ranking.idxmax()
print(f"Payment method with highest cancelled orders: {highest_cancelled_payment}")
print(payment_ranking)

Payment method with highest cancelled orders: credit_card
payment_method
apple_pay         5190
bank_transfer     2535
cod              15468
credit_card      28452
paypal            7817
dtype: int64


# Question 9

In [42]:
#how many orders by size
df_order_items_products = pd.merge(df_order_items, df_products, on = "product_id")
orders_by_size = df_order_items_products.groupby("size").size()
print(orders_by_size)
#how many orders are returned by size
returns_by_size = df_products_returns.groupby("size").size()
print(returns_by_size)

return_rate_by_size = (returns_by_size / orders_by_size)
return_rate_by_size

highest_return_rate = return_rate_by_size.idxmax()
print(f"The size with the highest return rate: {highest_return_rate}")

size
L     173174
M     176428
S     172042
XL    193025
dtype: int64
size
L      9741
M      9820
S      9723
XL    10655
dtype: int64
The size with the highest return rate: S


# Question 10

In [43]:
df_payments = pd.read_csv("/Users/doanhaidang1703/Documents/AIO/datathon 2026/data/payments.csv")
average_order_value = df_payments.groupby("installments")["payment_value"].mean()
highest_aov_installment = average_order_value.idxmax()
print(f"The installment with the highest AOV is: {highest_aov_installment}")

The installment with the highest AOV is: 6
